In [ ]:
# Load packages
import os

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

In [ ]:
# Git clone STAAR tutorial to current directory 
!git clone https://github.com/li-lab-genetics/STAARpipeline-Tutorial.git

In [ ]:
# Copy STAAR database split file to workspace bucket 
!gcloud storage cp STAARpipeline-Tutorial/FAVORannotator_csv/FAVORdatabase_chrsplit.csv ${WORKSPACE_BUCKET}/data/staar/axillary/

# COMMAND 

In [ ]:
%%writefile variant_info_gds.R
#!/usr/bin/env Rscript

##########################################################################
# INPUT
##########################################################################

args <- commandArgs(trailingOnly = TRUE)
# Read command-line arguments supplied to the R script.

# Expected arguments:
#   args[1] = DB split information CSV
#   args[2] = input GDS
#   args[3] = output directory
#   args[4] = chromosome number
file_DBsplit <- args[1]
gds_fn <- args[2]
output_dir <- args[3]
chr <- as.integer(args[4])

cat("DATABASE SPLIT:", file_DBsplit, "\n")
cat("GDS:", gds_fn, "\n")
cat("CHROMOSOME:", chr, "\n")
cat("OUTPUT:", output_dir, "\n")

###########################################################################
# MAIN FUNCTION
###########################################################################

### Load pacakges 
library(gdsfmt)
library(SeqArray)
library(SeqVarTools)

# Read the genomic block-definition CSV.
# DB_info is now a data frame containing something like:
#   Chr   Start_Pos   End_Pos
#   21    100000      200000
#   21    200001      300000
#   21    300001      400000
DB_info <- read.csv(file_DBsplit,header=TRUE)
DB_info <- DB_info[DB_info$Chr==chr,]

genofile <- seqOpen(gds_fn)

# Display the first five variants as a sanity check.
# This lets you verify that the GDS contains the expected:
#   chromosome
#   position
#   reference allele
#   alternate allele
CHR <- as.numeric(seqGetData(genofile, "chromosome"))
POSITION <- as.integer(seqGetData(genofile, "position"))
REF <- as.character(seqGetData(genofile, "$ref"))
ALT <- as.character(seqGetData(genofile, "$alt"))

cat("\nFirst 5 variants:\n")
print(
    data.frame(
        CHR = CHR[1:5],
        POSITION = POSITION[1:5],
        REF = REF[1:5],
        ALT = ALT[1:5]
    )
)

# Construct a unique-ish variant identifier for every variant.
# For example:
#   21-100123-A-T
#   21-100456-G-A
#   21-100789-C-G
VarInfo_genome <- paste0(CHR, "-", POSITION, "-", REF, "-", ALT)

# Print the first few generated variant identifiers as a sanity check.
cat("\nFirst 5 VarInfo_genome:\n")
print(head(VarInfo_genome))

# Close the GDS because all required variant information has now
# been extracted into R objects.
seqClose(genofile)

##########################################################################
# GENERATE BLOCK-SPECIFIC VARIANT INFORMATION
##########################################################################

for(kk in 1:dim(DB_info)[1])
{
	print(kk)

	VarInfo <- VarInfo_genome[(POSITION>=DB_info$Start_Pos[kk])&(POSITION<=DB_info$End_Pos[kk])]
	VarInfo <- data.frame(VarInfo)
	
	write.csv(VarInfo, paste0(output_dir,"/variant_info_chr",chr,"_block",kk,".csv"),
              quote=FALSE,
              row.names = FALSE)
}

In [ ]:
# Save script to workspace bucket 
!gcloud storage cp variant_info_gds.R ${WORKSPACE_BUCKET}/data/scripts/

In [ ]:
# Check script in workspace bucket 
!gcloud storage cat ${WORKSPACE_BUCKET}/data/scripts/variant_info_gds.R 

# SUBMIT COMMAND 

In [ ]:
import subprocess
import pandas as pd

# Create dataframe of size of exome plink files in GB 
chromosome = []
sizes = [] 

# Loop for all chromosome 
# for chrom in range(1,23):
for chrom in [2, 4, 11, 15, 17, 19, 21]:
    
    # Extract size of genomic plink bed file in GB for each chromosome 
    command = [
        "gcloud", "storage", "ls",
        "-l",
        f"{os.environ['WORKSPACE_BUCKET']}/data/staar/gds/chr{chrom}.gds",
        f"--billing-project={os.environ['GOOGLE_PROJECT']}"
    ]
    
    result = subprocess.run(command, capture_output=True, text=True)

    # Extract genomic size from string 
    lines = result.stdout.splitlines()
    line = lines[0].split()
    
    # Create list genomic size 
    sizes.append(line[0])
    chromosome.append(chrom)
    
# Deposit genomic file size into data frame 
genomic_size = pd.DataFrame({
    "chromosome": chromosome,
    "size": sizes
})

# Format genomic file size 
genomic_size['chromosome'] = genomic_size['chromosome'].astype('int')
genomic_size['size'] = (genomic_size['size'].astype('int') / 1e9) 

print(genomic_size)


In [ ]:
import shlex

# Logging directory
LOGGING = f"{os.environ['WORKSPACE_BUCKET']}/data/logging"

# Dsub script
SCRIPT = "variant_info_gds.R"
PATH = f"{os.environ['WORKSPACE_BUCKET']}/data/scripts/{SCRIPT}"

# Split database path 
SPLIT = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/axillary/FAVORdatabase_chrsplit.csv"

MAX_DISK_SIZE = 10 # Set maximum disk; adjust for acaf or exome accordingly
EXTRA_DISK_SIZE = 5 # Set additional disk size for each chromosome 

# Loop to submit dsub command
# for CHROM in range(1, 23):
for CHROM in [2, 4, 11, 15, 17, 19]:

    print(f"Submitting dsub command for chromosome {CHROM}")

    DISK_SIZE = (
        round(
            float(genomic_size.loc[genomic_size["chromosome"] == CHROM, "size"
                ].iloc[0]
            )
        )
        + EXTRA_DISK_SIZE
    )

    # REMOVE AFTER TEST RUN
    DISK_SIZE = 5

    print(f"Requesting disk size: {DISK_SIZE}")

    if DISK_SIZE > MAX_DISK_SIZE:
        print(f"Disk too large for chromosome {CHROM}")
        break

    if DISK_SIZE > 1000:
        print(f"Disk too large for chromosome {CHROM}")
        break

    # File paths
    GDS = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/gds/chr{CHROM}.gds"
    OUTPUT = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/variant_info/chr{CHROM}"

    # R script command
    R_COMMAND = 'Rscript "$SCRIPT" "$SPLIT" "$GDS" "$OUTPUT" "$CHROM"'
    print(R_COMMAND)
    command = [
        "aou_dsub",
            "--image", f"{os.environ['ARTIFACT_REGISTRY_DOCKER_REPO']}/vieno99/favor:v1.0",
            "--boot-disk-size", "5",
            "--disk-size", str(DISK_SIZE),
            "--machine-type", "n1-standard-1",
            "--logging", LOGGING,
            "--input", f"SCRIPT={PATH}",
            "--input", f"SPLIT={SPLIT}",
            "--input", f"GDS={GDS}",
            "--output-recursive", f"OUTPUT={OUTPUT}",
            "--env", f"CHROM={CHROM}",
            "--command", R_COMMAND,
    ]

    # Run in same shell as source command
    dsub_command = subprocess.run(
        f"source ~/aou_dsub.bash && {shlex.join(command)}",
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )

    print(dsub_command.stdout)
    print(dsub_command.stderr)

In [ ]:
# Create download folder 
!rm -rf download
!mkdir -p download 

In [ ]:
# Copy variant list to folder
!gcloud storage cp -r ${WORKSPACE_BUCKET}/data/staar/variant_info/* download

In [ ]:
# Compress download folder 
!tar -czvf download.tar.gz download/